In [3]:
import sys
from collections import defaultdict
from itertools import combinations


In [5]:
def parse_gr(path):
    n = None
    m = None
    edges = []

    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()

            if not line or line.startswith("c"):
                continue

            parts = line.split()

            if parts[0] == "p":
                if len(parts) != 4:
                    raise ValueError("Invalid p-line in .gr file")

                _, problem, n_str, m_str = parts

                if problem != "tw":
                    raise ValueError("Expected problem descriptor 'tw'")

                n = int(n_str)
                m = int(m_str)

            else:
                if n is None:
                    raise ValueError("Edge appeared before p-line")

                if len(parts) != 2:
                    raise ValueError("Invalid edge line in .gr file")

                u, v = map(int, parts)

                if not (1 <= u <= n and 1 <= v <= n):
                    raise ValueError("Vertex out of range in .gr file")

                edges.append((u, v))

    if n is None or m is None:
        raise ValueError("Missing p-line in .gr file")

    if len(edges) != m:
        raise ValueError(f"Expected {m} edges, got {len(edges)}")

    return n, edges


In [6]:
def parse_td(path):
    n_bags = None
    max_bag_size = None
    n_vertices = None

    bags = {}
    tree_edges = []

    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()

            if not line or line.startswith("c"):
                continue

            parts = line.split()

            if parts[0] == "s":
                if len(parts) != 5:
                    raise ValueError("Invalid s-line in .td file")

                _, problem, n_bags_str, max_bag_size_str, n_vertices_str = parts

                if problem != "td":
                    raise ValueError("Expected solution descriptor 'td'")

                n_bags = int(n_bags_str)
                max_bag_size = int(max_bag_size_str)
                n_vertices = int(n_vertices_str)

            elif parts[0] == "b":
                if n_bags is None:
                    raise ValueError("Bag appeared before s-line")

                bag_id = int(parts[1])

                if not (1 <= bag_id <= n_bags):
                    raise ValueError("Bag id out of range")

                bag_vertices = tuple(map(int, parts[2:]))

                bags[bag_id] = bag_vertices

            else:
                if n_bags is None:
                    raise ValueError("Tree edge appeared before s-line")

                if len(parts) != 2:
                    raise ValueError("Invalid tree edge line")

                a, b = map(int, parts)

                if not (1 <= a <= n_bags and 1 <= b <= n_bags):
                    raise ValueError("Tree edge contains invalid bag id")

                tree_edges.append((a, b))

    if n_bags is None:
        raise ValueError("Missing s-line in .td file")

    for i in range(1, n_bags + 1):
        if i not in bags:
            raise ValueError(f"Missing bag line for bag {i}")

    return n_bags, max_bag_size, n_vertices, bags, tree_edges


In [ ]:






def build_graph(n, edges):
    adj = [set() for _ in range(n + 1)]
    looped = set()

    for u, v in edges:
        if u == v:
            looped.add(u)
        else:
            adj[u].add(v)
            adj[v].add(u)

    return adj, looped


def build_tree(n_bags, tree_edges):
    tree = [[] for _ in range(n_bags + 1)]

    for a, b in tree_edges:
        tree[a].append(b)
        tree[b].append(a)

    if n_bags > 1 and len(tree_edges) != n_bags - 1:
        raise ValueError(".td tree must have exactly N - 1 edges")

    return tree


def root_tree(tree, root):
    parent = {root: 0}
    order = [root]

    for t in order:
        for x in tree[t]:
            if x == parent[t]:
                continue
            if x in parent:
                raise ValueError(".td tree contains a cycle")
            parent[x] = t
            order.append(x)

    if len(parent) != len(tree) - 1:
        raise ValueError(".td tree is disconnected")

    children = defaultdict(list)

    for x, p in parent.items():
        if p != 0:
            children[p].append(x)

    postorder = list(reversed(order))

    return parent, children, postorder


def mask_to_set(mask, bag):
    result = set()

    for i, v in enumerate(bag):
        if mask & (1 << i):
            result.add(v)

    return result


def set_weight(vertices, weights):
    return sum(weights[v] for v in vertices)


def is_independent(vertices, adj, looped):
    for v in vertices:
        if v in looped:
            return False

    for v in vertices:
        for u in adj[v]:
            if u in vertices:
                return False

    return True


def all_independent_masks(bag, adj, looped):
    masks = []

    for mask in range(1 << len(bag)):
        chosen = mask_to_set(mask, bag)

        if is_independent(chosen, adj, looped):
            masks.append(mask)

    return masks


def compatible(parent_mask, parent_bag, child_mask, child_bag):
    parent_choice = {}

    for i, v in enumerate(parent_bag):
        parent_choice[v] = bool(parent_mask & (1 << i))

    for j, v in enumerate(child_bag):
        if v in parent_choice:
            child_has_v = bool(child_mask & (1 << j))

            if parent_choice[v] != child_has_v:
                return False

    return True


def selected_overlap_weight(parent_mask, parent_bag, child_bag, weights):
    child_vertices = set(child_bag)
    total = 0

    for i, v in enumerate(parent_bag):
        if v in child_vertices and parent_mask & (1 << i):
            total += weights[v]

    return total


def maximum_weight_independent_set(n, edges, n_bags, bags, tree_edges, weights=None):
    if weights is None:
        weights = [0] + [1] * n

    adj, looped = build_graph(n, edges)
    tree = build_tree(n_bags, tree_edges)

    root = 1
    _, children, postorder = root_tree(tree, root)

    bag_masks = {}
    dp = {}
    choice = {}

    for t in postorder:
        bag = bags[t]
        valid_masks = all_independent_masks(bag, adj, looped)
        bag_masks[t] = valid_masks

        dp[t] = {}
        choice[t] = {}

        for mask in valid_masks:
            chosen_vertices = mask_to_set(mask, bag)
            value = set_weight(chosen_vertices, weights)
            child_choices = {}

            for child in children[t]:
                best_child_value = None
                best_child_mask = None

                for child_mask in bag_masks[child]:
                    if not compatible(mask, bag, child_mask, bags[child]):
                        continue

                    overlap = selected_overlap_weight(mask, bag, bags[child], weights)
                    candidate = dp[child][child_mask] - overlap

                    if best_child_value is None or candidate > best_child_value:
                        best_child_value = candidate
                        best_child_mask = child_mask

                if best_child_value is None:
                    raise ValueError("No compatible child state found")

                value += best_child_value
                child_choices[child] = best_child_mask

            dp[t][mask] = value
            choice[t][mask] = child_choices

    best_root_mask = max(dp[root], key=lambda mask: dp[root][mask])
    best_value = dp[root][best_root_mask]

    result = set()

    def reconstruct(t, mask):
        result.update(mask_to_set(mask, bags[t]))

        for child, child_mask in choice[t][mask].items():
            reconstruct(child, child_mask)

    reconstruct(root, best_root_mask)

    return best_value, result




In [7]:
def main():
    if len(sys.argv) != 3:
        print("Usage: python mwis_td.py graph.gr decomposition.td")
        sys.exit(1)

    gr_path = sys.argv[1]
    td_path = sys.argv[2]

    n, edges = parse_gr(gr_path)
    n_bags, _, td_n_vertices, bags, tree_edges = parse_td(td_path)

    if td_n_vertices != n:
        raise ValueError(".gr and .td disagree on number of graph vertices")

    value, independent_set = maximum_weight_independent_set(
        n=n,
        edges=edges,
        n_bags=n_bags,
        bags=bags,
        tree_edges=tree_edges,
    )

    print(value)
    print(" ".join(map(str, sorted(independent_set))))

